# Fraud-Score System - Run-All Demo

Builds and tests the whole polyglot pipeline. Just choose **Run > Run All Cells**.

Steps: check tools -> install Python deps -> build Rust engine -> run Rust tests -> direct engine check -> in-process Rust+FastAPI pipeline -> full live end-to-end with the Node worker -> show logs.

Place this notebook inside the `fraud-score-system` folder. Needs Rust (cargo), Python 3.10+, and Node 18+ on PATH. Cells that need a missing tool will SKIP (print a note) instead of crashing, so Run All always finishes.

## 1. Locate project + check tools

In [ ]:
import sys, os, shutil, subprocess, json, time, socket, importlib
from pathlib import Path

def find_base():
    here = Path.cwd()
    for cand in (here, here / 'fraud-score-system', here.parent):
        if (cand / 'engine').is_dir() and (cand / 'service').is_dir():
            return cand
    return None

BASE = find_base()
if BASE is None:
    print('ERROR: could not find the project. Put this notebook inside the fraud-score-system folder.')
else:
    ENGINE_DIR = BASE / 'engine'
    SERVICE_DIR = BASE / 'service'
    WORKER_DIR = BASE / 'worker'
    IS_WIN = os.name == 'nt'
    ENGINE_BIN = ENGINE_DIR / 'target' / 'release' / ('engine.exe' if IS_WIN else 'engine')
    PORT = 8011
    print('Project root:', BASE)

    def check(name):
        path = shutil.which(name)
        print(('  OK      ' if path else '  MISSING ') + name, '->', path)
        return path is not None

    print('Tools:')
    have_cargo = check('cargo')
    have_node = check('node')
    print('  python   ->', sys.executable)
    if not have_cargo:
        print('  (install Rust from https://rustup.rs to build the engine)')
    if not have_node:
        print('  (install Node 18+ from https://nodejs.org for the live worker demo)')

## 2. Install Python dependencies

In [ ]:
req = SERVICE_DIR / 'requirements.txt'
print('Installing from', req)
res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])
print('pip exit code:', res.returncode)
for mod in ('fastapi', 'httpx', 'pytest', 'uvicorn'):
    try:
        importlib.import_module(mod)
        print('  OK import', mod)
    except Exception as e:
        print('  FAIL import', mod, '->', e)

## 3. Build the Rust engine (release) + run its tests

In [ ]:
if not have_cargo:
    print('SKIP: cargo not found, cannot build the Rust engine.')
else:
    print('Building release binary (first build can take a minute)...')
    b = subprocess.run(['cargo', 'build', '--release'], cwd=ENGINE_DIR, capture_output=True, text=True)
    print(b.stdout)
    print(b.stderr)
    print('build exit code:', b.returncode)
    print('binary exists:', ENGINE_BIN.exists(), '->', ENGINE_BIN)
    print()
    print('Running cargo test...')
    t = subprocess.run(['cargo', 'test'], cwd=ENGINE_DIR, capture_output=True, text=True)
    print(t.stdout[-3000:])
    print(t.stderr[-1500:])
    print('cargo test exit code:', t.returncode)

## 4. Direct engine check (one transaction through the binary)

In [ ]:
sample = {'id': 1, 'amount': 12000.0, 'currency': 'INR', 'hour': 3, 'country': 'RU', 'status': 'pending'}
if not ENGINE_BIN.exists():
    print('SKIP: engine binary not built (run the build cell).')
else:
    p = subprocess.run([str(ENGINE_BIN)], input=json.dumps(sample), capture_output=True, text=True)
    print('stdout:', p.stdout.strip())
    print('stderr:', p.stderr.strip())
    print('exit code:', p.returncode)
    score = json.loads(p.stdout)
    assert score['score'] == 90, score
    print('PASS: engine scored 90 as expected.')

## 5. In-process pipeline (Rust + FastAPI, no live server)

Uses FastAPI's TestClient, so no server is started. Proves the contract: ingest -> pending -> real Rust binary -> submit score -> scored.

In [ ]:
if not ENGINE_BIN.exists():
    print('SKIP: engine binary missing - run the build cell first.')
else:
    if str(SERVICE_DIR) not in sys.path:
        sys.path.insert(0, str(SERVICE_DIR))
    from app.main import app, _reset
    from fastapi.testclient import TestClient

    _reset()
    client = TestClient(app)

    print('1) POST a transaction')
    created = client.post('/transactions', json={'amount': 12000, 'currency': 'INR', 'hour': 3, 'country': 'RU'}).json()
    print('   created:', created)
    tid = created['id']

    print('2) GET pending')
    pending = client.get('/transactions/pending').json()
    print('   pending:', pending)

    print('3) Run the Rust engine on the pending txn')
    p = subprocess.run([str(ENGINE_BIN)], input=json.dumps(pending[0]), capture_output=True, text=True)
    score = json.loads(p.stdout)
    print('   score:', score)

    print('4) Submit the score')
    client.post('/transactions/' + str(tid) + '/score', json=score)

    print('5) GET the transaction')
    final = client.get('/transactions/' + str(tid)).json()
    print('   final:', final)
    assert final['status'] == 'scored' and final['score'] == 90
    print('PASS: in-process pipeline works (Rust + FastAPI).')

## 6. Full live end-to-end (Rust + FastAPI + Node worker)

Starts a real uvicorn server and the Node worker as background processes, posts a transaction, and waits for the worker to score it. Everything is cleaned up in the `finally` block.

In [ ]:
import urllib.request

def http_post(url, payload):
    data = json.dumps(payload).encode()
    req = urllib.request.Request(url, data=data, headers={'Content-Type': 'application/json'}, method='POST')
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.loads(r.read().decode())

def http_get(url):
    with urllib.request.urlopen(url, timeout=5) as r:
        return json.loads(r.read().decode())

def port_open(port):
    s = socket.socket(); s.settimeout(0.3)
    try:
        s.connect(('127.0.0.1', port)); return True
    except OSError:
        return False
    finally:
        s.close()

base_url = 'http://127.0.0.1:' + str(PORT)
if not have_node:
    print('SKIP live demo: node not found. The in-process cell already proved Rust+FastAPI.')
elif not ENGINE_BIN.exists():
    print('SKIP live demo: engine binary not built.')
else:
    service_log = open(BASE / 'service.log', 'w')
    worker_log = open(BASE / 'worker.log', 'w')
    service_proc = None
    worker_proc = None
    try:
        print('Starting service on', base_url)
        service_proc = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'app.main:app', '--port', str(PORT)], cwd=SERVICE_DIR, stdout=service_log, stderr=subprocess.STDOUT)
        service_up = False
        for _ in range(40):
            if port_open(PORT):
                service_up = True
                break
            time.sleep(0.5)
        if not service_up:
            print('Service did not start - see service.log below.')
        else:
            print('Service is up. Starting Node worker...')
            env = dict(os.environ, SERVICE_URL=base_url, ENGINE_BIN=str(ENGINE_BIN))
            worker_proc = subprocess.Popen(['node', 'worker.js'], cwd=WORKER_DIR, stdout=worker_log, stderr=subprocess.STDOUT, env=env)
            time.sleep(2)

            print('Posting a transaction...')
            created = http_post(base_url + '/transactions', {'amount': 12000, 'currency': 'INR', 'hour': 3, 'country': 'RU'})
            tid = created['id']
            print('  created id', tid)

            print('Waiting for the worker to score it...')
            final = None
            for _ in range(20):
                time.sleep(1)
                final = http_get(base_url + '/transactions/' + str(tid))
                print('  status:', final['status'])
                if final['status'] == 'scored':
                    break
            assert final and final['status'] == 'scored' and final['score'] == 90, final
            print('PASS: full live end-to-end works (Rust + FastAPI + Node).')
            print('  final:', final)
    finally:
        for proc in (worker_proc, service_proc):
            if proc is not None:
                proc.terminate()
                try:
                    proc.wait(timeout=5)
                except Exception:
                    proc.kill()
        service_log.close()
        worker_log.close()
        print('Stopped service and worker.')

## 7. Logs (service + worker)

In [ ]:
for name in ('worker.log', 'service.log'):
    path = BASE / name
    if path.exists():
        print('===== ' + name + ' =====')
        print(path.read_text()[-2000:])
    else:
        print('(no ' + name + ' - live demo may have been skipped)')

## Troubleshooting

- **`cargo` / `node` MISSING**: install them, restart the kernel, Run All again.
- **Port 8011 in use**: change `PORT` in cell 1 to a free port and re-run.
- **`from app.main` import error**: make sure cell 2 (pip install) ran without errors.
- **Live demo SKIPPED**: that is fine - it only needs Node; the in-process cell (section 5) already proves the Rust+FastAPI pipeline.
- **Worker logs show 'engine not found'**: run the build cell (section 3) first so the release binary exists.